In [1]:
import io
import zipfile
import requests
import frontmatter
from tqdm.auto import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
from minsearch import Index, VectorSearch

## 1. Load Repository Data

In [3]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [4]:
moneyPrinterV2_docs = read_repo_data('FujiwaraChoki', 'MoneyPrinterV2')
print(f"MoneyPrinterV2 documents: {len(moneyPrinterV2_docs)}")

MoneyPrinterV2 documents: 10


## 2. Chunking with Sliding Window

In [5]:
def sliding_window(seq, size, step):
    """
    Split text into overlapping chunks.
    
    Args:
        seq: Input text string
        size: Size of each chunk in characters
        step: Step size between chunks (overlap = size - step)
    
    Returns:
        List of dicts with 'start' position and 'chunk' text
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [6]:
moneyPrinter_chunks = []

for doc in moneyPrinterV2_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    moneyPrinter_chunks.extend(chunks)

print(f"Total chunks: {len(moneyPrinter_chunks)}")

Total chunks: 24


## 3. Text-Based Index (BM25)

In [7]:
text_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

text_index.fit(moneyPrinter_chunks)

## 4. Vector-Based Index (Semantic Search)

In [8]:
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [9]:
moneyPrinter_embeddings = []

for d in tqdm(moneyPrinter_chunks):
    v = embedding_model.encode(d['chunk'])
    moneyPrinter_embeddings.append(v)

moneyPrinter_embeddings = np.array(moneyPrinter_embeddings)

  0%|          | 0/24 [00:00<?, ?it/s]

In [11]:
vector_index = VectorSearch()
vector_index.fit(moneyPrinter_embeddings, moneyPrinter_chunks)

## 5. Hybrid Search Implementation

In [12]:
def text_search(query, num_results=5):
    """
    Perform BM25-based keyword search.
    """
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    """
    Perform semantic vector search using embeddings.
    """
    query_embedding = embedding_model.encode(query)
    return vector_index.search(query_embedding, num_results=num_results)

def hybrid_search(query, num_results=5):
    """
    Combine text and vector search results.
    Returns deduplicated results from both approaches.
    """
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)
    
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        result_id = f"{result['filename']}_{result.get('start', 0)}"
        if result_id not in seen_ids:
            seen_ids.add(result_id)
            combined_results.append(result)
    
    return combined_results[:num_results * 2]

## 6. Test the Hybrid Search

In [14]:
query = "How to setup and use MoneyPrinterV2?"

print("=" * 80)
print(f"Query: {query}")
print("=" * 80)

results = hybrid_search(query, num_results=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"File: {result['filename']}")
    print(f"Title: {result.get('title', 'N/A')}")
    print(f"Start: {result.get('start', 0)}")
    print(f"Chunk preview: {result['chunk'][:200]}...")
    print("-" * 80)

Query: How to setup and use MoneyPrinterV2?

Result 1:
File: MoneyPrinterV2-main/CLAUDE.md
Title: N/A
Start: 0
Chunk preview: # CLAUDE.md

This file provides guidance to Claude Code (claude.ai/code) when working with code in this repository.

## Project Overview

MoneyPrinterV2 (MPV2) is a Python 3.12 CLI tool that automates...
--------------------------------------------------------------------------------

Result 2:
File: MoneyPrinterV2-main/README.md
Title: N/A
Start: 0
Chunk preview: # MoneyPrinter V2
 
> ♥︎ **Sponsor**: The Best AI Chat App: [shiori.ai](https://www.shiori.ai). Use code **MPV2** for 20% off.

---

> 𝕏 Also, follow me on X: [@DevBySami](https://x.com/DevBySami).

[...
--------------------------------------------------------------------------------

Result 3:
File: MoneyPrinterV2-main/README.md
Title: N/A
Start: 2000
Chunk preview: ould like to submit your own version/fork of MoneyPrinter, please open an issue describing the changes you made to the fork.

## Installa

## 7. Compare Search Methods

In [ ]:
def compare_search_methods(query):
    """
    Compare text, vector, and hybrid search results side by side.
    """
    print(f"Query: {query}")
    print("=" * 80)
    
    text_results = text_search(query, num_results=3)
    vector_results = vector_search(query, num_results=3)
    hybrid_results = hybrid_search(query, num_results=3)
    
    print(f"\n📝 TEXT SEARCH ({len(text_results)} results):")
    for i, r in enumerate(text_results, 1):
        print(f"{i}. {r['filename']} - {r.get('title', 'N/A')}")
    
    print(f"\n🔍 VECTOR SEARCH ({len(vector_results)} results):")
    for i, r in enumerate(vector_results, 1):
        print(f"{i}. {r['filename']} - {r.get('title', 'N/A')}")
    
    print(f"\n🔀 HYBRID SEARCH ({len(hybrid_results)} results):")
    for i, r in enumerate(hybrid_results, 1):
        print(f"{i}. {r['filename']} - {r.get('title', 'N/A')}")

In [ ]:
compare_search_methods("How to install and configure the application?")

In [ ]:
compare_search_methods("video generation features")